# astrowidgets image display demo

This runs entirely in your browser (xeus-python/WebAssembly kernel) with
`astrowidgets` installed from the tip of `main` on `astropy/astrowidgets`.

**Note:** only the **bqplot** backend works in the browser — the ginga backend
needs `aggdraw`, a compiled C extension that is not available for WASM.

In [ ]:
import numpy as np
from astropy.table import Table
import astropy.visualization as apviz

from astrowidgets.bqplot import ImageWidget

Generate a synthetic star field: Gaussian "stars" on a noisy background
(no network access is needed in the WASM kernel).

In [ ]:
rng = np.random.default_rng(42)

ny, nx = 500, 500
n_stars = 50

image = rng.normal(loc=100, scale=5, size=(ny, nx))

x_stars = rng.uniform(20, nx - 20, n_stars)
y_stars = rng.uniform(20, ny - 20, n_stars)
fluxes = 10 ** rng.uniform(3, 5.5, n_stars)
fwhm = 4.0
sigma = fwhm / 2.355

yy, xx = np.mgrid[0:ny, 0:nx]
for x0, y0, flux in zip(x_stars, y_stars, fluxes):
    image += flux / (2 * np.pi * sigma**2) * np.exp(
        -((xx - x0) ** 2 + (yy - y0) ** 2) / (2 * sigma**2)
    )

image = rng.poisson(image).astype(float)

Create the widget and load the image. Scroll to zoom, drag to pan.

In [ ]:
imw = ImageWidget()
imw.load_image(image)
imw

Adjust cuts, stretch and colormap — the display above updates in place.

In [ ]:
imw.set_cuts(apviz.AsymmetricPercentileInterval(1, 99.5))
imw.set_stretch(apviz.AsinhStretch(0.1))
imw.set_colormap('viridis')

Overlay markers at the known star positions.

In [ ]:
stars = Table({'x': x_stars, 'y': y_stars})
imw.load_catalog(
    stars,
    catalog_label='stars',
    catalog_style={'color': 'red', 'shape': 'circle', 'size': 60},
)